In [2]:
import os
from langchain_ollama import OllamaLLM
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
from langchain import hub

# 1. Ollama를 통한 Gemma 4 설정
# WSL 환경이라면 base_url에 윈도우 호스트 IP를 넣어야 할 수도 있습니다.
llm = OllamaLLM(
    model="gemma4:26b", 
    temperature=0,
    base_url="http://172.27.48.1:11434", # 기본값
    timeout=900,
    model_kwargs={
        "num_gpu": 99,       # 가능한 모든 레이어를 GPU로 보내라는 신호
        "main_gpu": 0,       # 3070(0번 GPU)을 주 연산 장치로 지정
    }
)

# 2. 도구(Tools) 정의
def read_file(path):
    try:
        with open(path.strip(), 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e: return f"에러: {e}"

def write_file(input_str):
    try:
        path, content = input_str.split("||", 1)
        with open(path.strip(), 'w', encoding='utf-8') as f:
            f.write(content)
        return f"{path} 저장 완료"
    except Exception as e: return f"에러: {e}"

tools = [
    Tool(name="ReadFile", func=read_file, description="파일 내용을 읽습니다. 인자: 파일경로"),
    Tool(name="WriteFile", func=write_file, description="파일을 씁니다. 인자: '파일명||내용'")
]

# 3. 에이전트 생성
# ReAct 방식은 gemma4:26b 정도 체급이면 아주 잘 수행합니다.
prompt = hub.pull("hwchase17/react")
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    handle_parsing_errors=True
)

print("✅ Gemma 4 에이전트 준비 완료!")

# 실행 테스트
# agent_executor.invoke({"input": "test.py 파일을 만들어서 'print(123)' 이라고 적어줘."})

/home/lamph/miniconda3/envs/solo/lib/python3.11/site-packages/langsmith/client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


✅ Gemma 4 에이전트 준비 완료!


In [3]:
agent_executor.invoke({"input":
"""
test.py 파일을 참조해서 flask를 사용해서 웹앱으로 구성해줘
app.py파일로 만들어주면돼
"""
})



> Entering new AgentExecutor chain...
Action: ReadFile
Action Input: test.pyimport os
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

def run_local_rag():
    # 1. 파일 경로 설정 (사용자가 제공한 파일명)
    pdf_files = [
        "국가연구개발사업 연구개발비 사용 기준.pdf",
        "국가연구개발혁신법.pdf",
        "국가 R&D 부적정 사례집.pdf"
    ]

    # 파일 존재 여부 확인
    for pdf in pdf_files:
        if not os.path.exists(pdf):
            print(f"Error: 파일을 찾을 수 없습니다 -> {pdf}")
            return

    print("1. PDF 문서 로딩 중...")
    documents = []
    for pdf in pdf_files:
        loader = PyPDFLoader(pdf)
        documents.append(loader.load())
    
    # 리스트 형태의 문서를 하나로 합침
    all_docs = []
    for doc_list in documents:
  

KeyboardInterrupt: 